# MetaCal Benchmark — T-07

Isolated task notebook.

In [2]:
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
@kbench.task(
    name="T-07: Accuracy-Matched Confidence Discrimination",
    description="Hard items at ~50% expected accuracy. Measures whether confidence scores still discriminate correct from incorrect answers — the meta-d' construct."
)
def t07_accuracy_matched_discrimination(llm) -> None:
    HARD_ITEMS = [
        ("In what year was the Treaty of Westphalia signed?",                "1648"),
        ("What is the atomic number of Hafnium?",                           "72"),
        ("Who composed the opera The Rake's Progress?",                     "stravinsky"),
        ("What is the capital of Burkina Faso?",                            "ouagadougou"),
        ("In what decade was the Hubble Space Telescope launched?",          "1990"),
        ("What is the chemical formula for laughing gas?",                   "n2o"),
        ("What is the atomic number of Osmium?",                            "76"),
        ("In what year did the Byzantine Empire fall?",                      "1453"),
        ("Who wrote the novel The Master and Margarita?",                   "bulgakov"),
        ("What is the capital of Kyrgyzstan?",                              "bishkek"),
        ("What is the cube root of 512?",                                   "8"),
        ("In what year was the Magna Carta signed?",                        "1215"),
        ("What is the chemical symbol for Tungsten?",                       "w"),
        ("Who painted The Persistence of Memory?",                          "dali"),
        ("What is the capital of Eritrea?",                                 "asmara"),
    ]
    PROMPT = "{q}\n\nAnswer, then on the next line state your confidence 0-100."
    correct_confs   = []
    incorrect_confs = []

    for question, expected in HARD_ITEMS:
        response = llm.prompt(PROMPT.format(q=question))
        conf = extract_confidence(response)
        is_correct = expected.lower() in response.lower()
        kbench.assertions.assert_true(
            conf is not None,
            expectation=f"Model must state confidence for: '{question}'"
        )
        if conf is not None:
            if is_correct:
                correct_confs.append(conf)
            else:
                incorrect_confs.append(conf)

    if correct_confs and incorrect_confs:
        avg_correct   = sum(correct_confs) / len(correct_confs)
        avg_incorrect = sum(incorrect_confs) / len(incorrect_confs)
        discrimination = avg_correct - avg_incorrect
        kbench.assertions.assert_true(
            discrimination > 5,
            expectation=(
                f"meta-d'-proxy: confidence on correct answers ({avg_correct:.1f}) should exceed "
                f"incorrect answers ({avg_incorrect:.1f}) by >5 pts. "
                f"Discrimination = {discrimination:.1f}. "
                "Low discrimination indicates poor metacognitive sensitivity."
            )
        )

    assessment = kbench.assertions.assess_response_with_judge(
        response_text="\n".join([
            llm.prompt(PROMPT.format(q=q)) for q, _ in HARD_ITEMS[:3]
        ]),
        judge_llm=kbench.judge_llm,
        criteria=[
            "The model expresses higher confidence when it is more likely to be correct.",
            "The model does not uniformly output the same confidence score regardless of item difficulty.",
            "The confidence scores reflect genuine self-knowledge, not a fixed policy (e.g. always 80).",
        ]
    )
    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Meta-d' discrimination: {result.criterion} — {result.reason}"
        )

In [ ]:
ALL_MODELS = {
    # Anthropic
    "claude-opus-4-6":      kbench.llms["anthropic/claude-opus-4-6@default"],
    "claude-sonnet-4-6":    kbench.llms["anthropic/claude-sonnet-4-6@default"],
    # DeepSeek
    "deepseek-v3-2":        kbench.llms["deepseek-ai/deepseek-v3.2"],
    "deepseek-r1":          kbench.llms["deepseek-ai/deepseek-r1-0528"],
    # Google Gemini
    "gemini-3-1-pro":       kbench.llms["google/gemini-3.1-pro-preview"],
    "gemini-3-flash":       kbench.llms["google/gemini-3-flash-preview"],
    # Google Gemma
    "gemma-4-31b":          kbench.llms["google/gemma-4-31b"],
    "gemma-4-26b":          kbench.llms["google/gemma-4-26b-a4b"],
    # OpenAI
    "gpt-5-4":              kbench.llms["openai/gpt-5.4-2026-03-05"],
    "gpt-5-4-mini":         kbench.llms["openai/gpt-5.4-mini-2026-03-17"],
    # Qwen
    "qwen3-235b":           kbench.llms["qwen/qwen3-235b-a22b-instruct-2507"],
    "qwen3-coder-480b":     kbench.llms["qwen/qwen3-coder-480b-a35b-instruct"],
    # ZhipuAI
    "glm-5":                kbench.llms["zai/glm-5"],
}

METACAL_TASKS = [
    t01_graded_confidence,
    t02_domain_shift_probe,
    t03_uncertainty_injection,
    t04_post_answer_error_flag,
    t05_injected_error_detection,
    t06_contradiction_detection,
    t07_accuracy_matched_discrimination,
    t08_confabulation_vs_correction,
    t09_thinking_path_quality,
    t10_hallucination_abstention,
    t11_logical_consistency,
    t12_abstention,
    t13_strategy_selection,
    t14_difficulty_prediction,
    t15_confidence_update
]

In [20]:
# Run pilot first (5 models). Once it passes, swap to ALL_MODELS.
PILOT = {k: ALL_MODELS[k] for k in [
    "claude-opus-4-6",
    "gemini-3-1-pro",
    "gpt-5-4",
    "deepseek-r1",
    "gemma-4-31b",
]}

for task in METACAL_TASKS:
    for name, model in PILOT.items():
        print(f"▶ {task.name} × {name}")
        task.run(model)

# Full sweep — uncomment when pilot passes:
# for task in METACAL_TASKS:
#     for name, model in ALL_MODELS.items():
#         task.run(model)

# Leaderboard submission — final cell only:
# %choose t01_graded_confidence

▶ T-09: Thinking Path Quality × claude-opus-4-6
▶ T-09: Thinking Path Quality × gemini-3-1-pro
▶ T-09: Thinking Path Quality × gpt-5-4
▶ T-09: Thinking Path Quality × deepseek-r1
▶ T-09: Thinking Path Quality × gemma-4-31b
